# LinguoMT — AfricaS2T Experiment Runner

Run any of the 4 experiment scripts in **debug** or **full** mode.

| Experiment | Model | Dataset | Languages |
|---|---|---|---|
| `FLEURS__SeamlessM4Tv2` | SeamlessM4T-v2-large | FLEURS | Igbo, Yoruba, Swahili |
| `FLEURS__WhisperNLLB` | Whisper-large-v3 + NLLB-600M | FLEURS | Yoruba, Swahili, Hausa |
| `AfricanCeltic__SeamlessM4Tv2` | SeamlessM4T-v2-large | African-Celtic | Igbo, Yoruba |
| `AfricanCeltic__WhisperNLLB` | Whisper-large-v3 + NLLB-600M | African-Celtic | Yoruba, Hausa |

**Before running:** set the runtime to **GPU** → Runtime → Change runtime type → T4 GPU (or A100 if available).

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Clone / Update Repository

In [ ]:
import os, subprocess

REPO_DIR = "/content/LinguoMT-AfricaS2T"

if os.path.exists(REPO_DIR):
    r = subprocess.run(["git", "-C", REPO_DIR, "pull"], capture_output=True, text=True)
    print("Repo updated:", r.stdout.strip() or r.stderr.strip())
else:
    REPO_URL = "https://github.com/prsisda/LinguoMT-AfricaS2T.git"
    r = subprocess.run(["git", "clone", REPO_URL, REPO_DIR], capture_output=True, text=True)
    print("Cloned:", r.stderr.strip() or "done")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## Step 3 — Select Experiment and Mode

In [ ]:
import re, pathlib

# ── EDIT THESE TWO LINES ────────────────────────────────────────────────
EXPERIMENT = "FLEURS__SeamlessM4Tv2"   # options: see table above
DEBUG_MODE = True                       # True ≈ 10 min GPU | False ≈ 30-60 min GPU
# ────────────────────────────────────────────────────────────────────────

VALID_EXPERIMENTS = [
    "FLEURS__SeamlessM4Tv2",
    "FLEURS__WhisperNLLB",
    "AfricanCeltic__SeamlessM4Tv2",
    "AfricanCeltic__WhisperNLLB",
]
assert EXPERIMENT in VALID_EXPERIMENTS, f"EXPERIMENT must be one of: {VALID_EXPERIMENTS}"

script_path = pathlib.Path(f"{EXPERIMENT}/notebooks/run_experiment.py")
assert script_path.exists(), f"Script not found: {script_path}"

mode_str = "True" if DEBUG_MODE else "False"
patched = re.sub(
    r"(?m)^(DEBUG_MODE\s*=\s*)(True|False)",
    rf"\g<1>{mode_str}",
    script_path.read_text()
)
script_path.write_text(patched)

print(f"Experiment : {EXPERIMENT}")
print(f"Mode       : {'DEBUG  (fast test)' if DEBUG_MODE else 'FULL   (paper run)'}")
print(f"Script     : {script_path}")

## Step 4 — Run Experiment

In [ ]:
!git -C /content/LinguoMT-AfricaS2T pull origin main

In [ ]:
script = str(script_path)
!python "$script"

---
## (Optional) Run All 4 Experiments Sequentially

Set `DEBUG_MODE_ALL` and run this cell to execute all 4 scripts back-to-back.
Results for each experiment are saved before the next one starts.

| Mode | Expected GPU time |
|------|------------------|
| DEBUG | ~40 min total (4 × ~10 min) |
| FULL  | ~2–3 hrs total (4 × 30–60 min) |

In [ ]:
import re, pathlib, subprocess, sys

# ── EDIT THIS LINE ───────────────────────────────────────────────────────
DEBUG_MODE_ALL = True   # True = debug run | False = full paper run
# ────────────────────────────────────────────────────────────────────────

ALL_EXPERIMENTS = [
    "FLEURS__SeamlessM4Tv2",
    "FLEURS__WhisperNLLB",
    "AfricanCeltic__SeamlessM4Tv2",
    "AfricanCeltic__WhisperNLLB",
]

mode_str = "True" if DEBUG_MODE_ALL else "False"

for exp in ALL_EXPERIMENTS:
    sp = pathlib.Path(f"{exp}/notebooks/run_experiment.py")
    patched = re.sub(
        r"(?m)^(DEBUG_MODE\s*=\s*)(True|False)",
        rf"\g<1>{mode_str}",
        sp.read_text()
    )
    sp.write_text(patched)

    print(f"\n{'='*60}")
    print(f"  Running: {exp}  [{'DEBUG' if DEBUG_MODE_ALL else 'FULL'}]")
    print(f"{'='*60}\n")

    result = subprocess.run([sys.executable, str(sp)])
    if result.returncode != 0:
        print(f"ERROR in {exp} (exit code {result.returncode}) — continuing to next...")

print("\nAll experiments finished.")

---
## (Optional) Save Results to Google Drive

Run this cell after your experiment(s) finish to collect all results into a single
timestamped folder in `MyDrive/LinguoMT-AfricaS2T/`.

Folder name: `LinguoMT-AfricaS2T-DEBUG-<TIMESTAMP>` (debug) or `LinguoMT-AfricaS2T-<TIMESTAMP>` (full).

In [ ]:
import shutil, glob
from datetime import datetime
from pathlib import Path

timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
drive_base = Path("/content/drive/MyDrive/LinguoMT-AfricaS2T")
drive_base.mkdir(parents=True, exist_ok=True)

# Find all Results_* folders written by the scripts in /content/
result_dirs = sorted(glob.glob("/content/Results_*"))

if not result_dirs:
    print("No Results_* folders found in /content/ — run experiments first.")
else:
    for src in result_dirs:
        results_name = Path(src).name          # e.g. Results_FLEURS__SeamlessM4Tv2_Large_DEBUG
        folder_name  = f"{results_name}-{timestamp}"
        drive_dest   = drive_base / folder_name
        if drive_dest.exists():
            shutil.rmtree(str(drive_dest))
        shutil.copytree(src, str(drive_dest))
        print(f"Saved: {folder_name}")
    print(f"\nAll results saved to:\n  {drive_base}")